# Explore: Amtrak State Supported Michigan Service

## Intercity Passenger Rail Service Station Performance Metrics

The Amtrak [network](https://www.amtrak.com/content/dam/projects/dotcom/english/public/documents/Maps/Amtrak-System-Map-020923.pdf)
is a passenger rail service that provides intercity rail service in the
continental United States and to select Canadian cities. The network is operated by the
[National Railroad Passenger Corporation](https://railroads.dot.gov/passenger-rail/amtrak/amtrak),
a federally chartered for-profit corporation that receives some state funding and covers its
operating costs by selling tickets and providing other services.

This notebook commences exploration of the augmented quarterly
[Amtrak](https://www.amtrak.com/home.html) station performance metrics for trains supported by
the State of Michigan. The goal is to better understand individual Amtrak Michigan Service
performance and identify potential areas for further analysis.

### Variable names

A number of variable names in this project leverage the following abbreviations. The naming
strategy is to strike a balance between brevity and readability:

* `amtk`: Amtrak (reporting mark)
* `chrt`: chart
* `cols`: columns
* `const`: constant
* `cwd`: current working directory
* `eb`: eastbound direction of travel
* `lm`: linear model
* `mi`: miles
* `mm`: minutes (ISO 8601)
* `nb`: northbound direction of travel
* `psgr`: passenger
* `qtr`: quarter
* `rte`: route
* `sb`: southbound direction of travel
* `stats`: summary statistics
* `stn`: station
* `stns`: stations
* `svc`: service
* `trn`: train
* `wb`: westbound direction of travel

In [1]:
import json
import numpy as np
import pandas as pd
import pathlib as pl
import tomllib as tl

import fra_amtrak.amtk_detrain as detrn
import fra_amtrak.amtk_frame as frm
import fra_amtrak.amtk_network as ntwk
import fra_amtrak.chart.chart as chrt
import fra_amtrak.chart.builders.bxplt_preagg_bldr as bxp_bld
import fra_amtrak.chart.builders.hstgrm_bldr as hst_bld
import fra_amtrak.chart.builders.hstgrm_lyr_bldr as hstl_bld
import fra_amtrak.chart.builders.line_bldr as ln_bld
import fra_amtrak.chart.schemas.bxplt_preagg as bxp
import fra_amtrak.chart.schemas.hstgrm as hst
import fra_amtrak.chart.schemas.hstgrm_lyr as hstl
import fra_amtrak.chart.schemas.line as ln

In [2]:
import warnings, numpy as np, scipy.stats as stats
import fra_amtrak.amtk_detrain as detrn
import fra_amtrak.amtk_frame as frm
import fra_amtrak.amtk_network as ntwk
import pandas as pd

def compute_sum_stats(frame, agg_columns, agg_funcs, precision=4):
    agg_dict = {col: agg_funcs for col in frame.columns if col in agg_columns}
    return frame.agg(agg_dict).round(precision)
detrn.compute_sum_stats = compute_sum_stats

def compute_sum_stats_by_group(frame, groups, agg_columns, agg_funcs, precision=4):
    agg_dict = {col: agg_funcs for col in frame.columns if col in agg_columns}
    return frame.groupby(groups).agg(agg_dict).round(precision)
detrn.compute_sum_stats_by_group = compute_sum_stats_by_group

def describe_numeric_column(column):
    try:
        if np.issubdtype(column.dtype, np.number):
            min_ = column.min(); max_ = column.max()
            return {
                "type": column.__class__, "name": column.name,
                "values": {"non_null": column.count(), "missing": column.isna().sum(), "dtype": column.dtype},
                "center": {"mean": column.mean(), "median": column.median(), "mode": column.mode()[0] if not column.mode().empty else None},
                "position": {"min": np.float64(min_), "25%": np.float64(column.quantile(0.25)), "50%": np.float64(column.quantile()), "75%": np.float64(column.quantile(0.75)), "max": np.float64(max_)},
                "spread": {"variance": column.var(), "std": column.std(), "range": max_ - min_, "iqr": stats.iqr(column)},
                "shape": {"skewness": column.skew(), "kurtosis": column.kurt()}
            }
    except Exception as e:
        warnings.warn(str(e), UserWarning)
    return None
frm.describe_numeric_column = describe_numeric_column

def bin_data(frame, column, bins):
    binned_data = frame.copy()
    binned_data["bin"] = pd.cut(binned_data[column], bins=bins, labels=False, include_lowest=True)
    binned_data = binned_data.dropna(subset=["bin"]).copy()
    binned_data["bin"] = binned_data["bin"].astype(int)
    binned_data["bin_start"] = binned_data["bin"].apply(lambda x: bins[x])
    binned_data["bin_end"] = binned_data["bin"].apply(lambda x: bins[x + 1] if x + 1 < len(bins) else bins[x])
    binned_data["bin_center"] = (binned_data["bin_start"] + binned_data["bin_end"]) / 2
    return binned_data
frm.bin_data = bin_data

def by_train_number(frame, train_number):
    return frame[frame[COLS["trn"]] == train_number].reset_index(drop=True)
ntwk.by_train_number = by_train_number

def by_sub_service(frame, sub_service):
    mask = frame[COLS["sub_svc"]].str.lower() == sub_service.lower()
    return frame[mask].reset_index(drop=True)
ntwk.by_sub_service = by_sub_service

print("✓ Semua patch berhasil!")

✓ Semua patch berhasil!


In [8]:
def format_year_quarter(row):
    return f"{int(row[COLS['year']])}Q{int(row[COLS['quarter']])}"
detrn.format_year_quarter = format_year_quarter

def assign_color(fiscal_quarter, colors=None):
    if colors is None:
        colors = [COLORS["amtk_blue"], COLORS["amtk_red"]]
    if fiscal_quarter is None or (isinstance(fiscal_quarter, float) and np.isnan(fiscal_quarter)):
        return colors[0]
    return colors[0] if int(str(fiscal_quarter)[-1]) % 2 == 0 else colors[1]
detrn.assign_color = assign_color
print("✓ Patch tambahan berhasil!")

✓ Patch tambahan berhasil!


In [10]:
def assign_color(fiscal_quarter, colors=None):
    if colors is None:
        colors = [COLORS["amtk_blue"], COLORS["amtk_red"]]
    if fiscal_quarter is None or (isinstance(fiscal_quarter, float) and np.isnan(fiscal_quarter)):
        return colors[0]
    try:
        return colors[0] if int(str(fiscal_quarter)[-1]) % 2 == 0 else colors[1]
    except:
        return colors[0]
detrn.assign_color = assign_color

## 1.0 Read files

### 1.1 Resolve paths

In [4]:
parent_path = pl.Path.cwd()  # current working directory
parent_path

PosixPath('/home/jovyan/work/assignments/Course4')

### 1.2 Load constants

Load a companion [TOML](https://toml.io/en/) file named `notebook.toml` containing constants.

In [5]:
filepath = parent_path.joinpath("notebook.toml")
with open(filepath, "rb") as file_obj:
    const = tl.load(file_obj)

# Access constants
AGG = const["agg"]
CHRT_BAR = const["chart"]["bar"]
COLORS = const["colors"]
COLS = const["columns"]
DIRECTION = const["train"]["direction"]
SVC = const["services"]
SUB_SVC = const["train"]["sub_service"]
TRN = const["train"]

### 1.3 Retrieve sub service route information

The file `amtk_sub_services.json` contains miscellaneous information about Amtrak sub services (i.e., named trains).

In [6]:
filepath = parent_path.joinpath("data", "processed", "amtk_sub_services.json")
with open(filepath, "r") as file:
    amtk_sub_svcs = json.load(file)
len(amtk_sub_svcs)

48

In [7]:
for sub_svc in amtk_sub_svcs:
    name = sub_svc.get("sub service", "")
    if any(x in name.lower() for x in ["blue water", "pere marquette", "wolverine"]):
        print(f"\n=== {name} ===")
        for k, v in sub_svc.items():
            if isinstance(v, dict):
                print(f"  '{k}': keys = {list(v.keys())}")
            else:
                print(f"  '{k}': {type(v).__name__}")


=== Blue Water ===
  'service line': str
  'sub service': str
  'url': str
  'hosts': list
  'station codes': list
  'station order': keys = ['eastbound', 'westbound']

=== Pere Marquette ===
  'service line': str
  'sub service': str
  'url': str
  'hosts': list
  'station codes': list
  'station order': keys = ['eastbound', 'westbound']

=== Wolverine ===
  'service line': str
  'sub service': str
  'url': str
  'hosts': list
  'station codes': list
  'station order': keys = ['eastbound', 'westbound']


### 1.4 Retrieve station details

The file `amtk_stations.csv` contains location-related information for all Amtrak stations.

In [ ]:
filepath = parent_path.joinpath("data", "processed", "amtk_stations.csv")
stations = pd.read_csv(filepath, dtype={"ZIP Code": "str"}, low_memory=False)
stations.shape

### 1.5 Retrieve performance data

In [ ]:
filepath = parent_path.joinpath("data", "processed", "station_performance_metrics-v1p2.csv")
trains = pd.read_csv(
    filepath, dtype={"Address 02": "str", "ZIP Code": "str"}, low_memory=False
)  # avoid DtypeWarning
trains.shape

### 1.6 Retrieve late time predictions

In [ ]:
filepath = parent_path.joinpath("data", "student", "stu-amtk-avg_min_late_predict.csv")
predictions = pd.read_csv(filepath, low_memory=False)
predictions.shape

## 2.0 State Supported Michigan Service [1 pt]

Amtrak's state-supported trains are funded by state governments. These services are typically
shorter in length and operate within a single state or across multiple states. Amtrak's
[Michigan](https://www.amtrak.com/michigan-services-train) service include the _Pere Marquette_,
_Blue Water_, and _Wolverine_ trains with routes between Chicago and Grand Rapids, Chicago and Port
Huron, and Chicago and greater Detroit.

Retrieve the Michigan Service performance data by calling the appropriate `amtk_network`
function. Assign the return value of the function call to a variable named `mich`.

In [9]:
# YOUR CODE HERE
mich = pd.concat([
    ntwk.by_sub_service(trains, SUB_SVC["blwtr"]),
    ntwk.by_sub_service(trains, SUB_SVC["prmrq"]),
    ntwk.by_sub_service(trains, SUB_SVC["wolv"]),
]).reset_index(drop=True)
mich.shape

NameError: name 'trains' is not defined

In [ ]:
# hidden tests are within this cell

### 2.1 Michigan service: on-time performance metrics (entire period) [1 pt]

Michigan service performance data is a compilation of quarterly metrics that focus on late
detraining passengers. Detraining passengers are considered on-time if they arrive at their
destination no later than fifteen (`15`) minutes after their scheduled arrival time. All other
detraining passengers are considered late.

In [ ]:
# YOUR CODE HERE
mich_trn_arrivals = mich.shape[0]
mich_detrn = mich[COLS["total_detrn"]].sum()
mich_detrn_late = mich[COLS["late_detrn"]].sum()
mich_detrn_on_time = mich_detrn - mich_detrn_late
print(f"Train Arrivals: {mich_trn_arrivals}", f"Total Detraining Customers: {mich_detrn}",
      f"Late Detraining Customers: {mich_detrn_late}", f"On-Time Detraining Customers: {mich_detrn_on_time}", sep="\n")
mich_stats = detrn.get_sum_stats(mich, AGG["columns"], AGG["funcs"])
mich_stats

In [ ]:
# hidden tests are within this cell

### 2.2 Michigan service: mean late arrival times [1 pt]

Review the central tendency, dispersion, and shape for the mean late arrival times of _Wolverine_ trains.

In [ ]:
# Drop missing values
mich_avg_mm_late = mich[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

mich_avg_mm_late.head(3)

# Call the custom frm.describe_numeric_column() function again
mich_avg_mm_late_describe = frm.describe_numeric_column(mich_avg_mm_late)
mich_avg_mm_late_describe

The skewness and kurtosis values returned suggest that the distribution of mean late arrival times of _Wolverine_ trains is positively skewed and features a sharper peak and heavier right tail than a normal distribution. Let's confirm this visually by generating a histogram.

In [ ]:
# hidden tests are within this cell

### 2.3 Michigan service: visualize distribution of mean late arrival times [1 pt]

Visualize mean late arrival times for the entire period. The data is binned prior to plotting.

#### 2.3.1 Create the chart data

In [ ]:
# Convert to DataFrame
mich_avg_mm_late = mich_avg_mm_late.to_frame(name=COLS["avg_mm_late"])

mich_mu = mich_avg_mm_late_describe["center"]["mean"] # Get mean and standard deviation
mich_sigma = mich_avg_mm_late_describe["spread"]["std"]

# Get max value (for x-axis ticks); pad max value for chart display
mich_max_val = mich_avg_mm_late_describe["position"]["max"]
mich_max_val_ceil = (np.ceil(mich_max_val / 10) * 10).astype(int)

# Create bins
mich_mm_late, mich_bins, mich_num_bins, mich_bin_width = frm.create_bins(
    mich_avg_mm_late, COLS["avg_mm_late"], 5
)

# Bin the data
chrt_data = frm.bin_data(mich_mm_late, COLS["avg_mm_late"], mich_bins)
# chrt_data

In [ ]:
# hidden tests are within this cell

#### 2.3.2 Generate the histogram

In [ ]:
title = chrt.format_title(mich_stats, f"Amtrak {SVC['mich']} Service Late Detraining Passengers")
chart = (
    hst_bld.HistogramBuilder(hst.HistogramChart(frame=chrt_data, title=title))
    .x(
        axis_tick_count=mich_max_val_ceil,
        axis_values=np.arange(0.0, mich_max_val_ceil + 15, 5).tolist(),
        bin_max_bins=mich_num_bins,
        bin_step=mich_bin_width,
        scale_domain=[0.0, mich_max_val_ceil],
    )
    .mu_rule(mu=mich_mu)
    .sigma_rules(sigma=mich_sigma, n=2)
    .build()
)
chart.display()

## 3.0 Michigan sub services: on-time performance metrics (entire period) [1 pt]

Call the appropriate `amtk_detrain` function and pass it the arguments required to return Michigan
service summary statistics grouped by sub service. Assign the return value of the function call to a
variable named `mich_sub_svcs_stats`.

In [ ]:
# YOUR CODE HERE
mich_sub_svcs_stats = detrn.get_sum_stats_by_group(mich, COLS["sub_svc"], AGG["columns"], AGG["funcs"])
mich_sub_svcs_stats

In [ ]:
# hidden tests are within this cell

### 3.1 Michigan sub services: visualize distribution of mean late arrival times

Visualize mean late arrival times for the entire period. The data is binned prior to plotting.

#### 3.1.1 Retrieve each sub service [3 pts]

Call the appropriate `amtk_network` function to retrieve the performance data for each
Michigan sub service (_Blue Water_, _Pere Marquette_, and _Wolverine_). Assign the return value of
each function call to the following variables:

1. _Blue Water_: `blwtr`
2. _Pere Marquette_: `prmrq`
3. _Wolverine_: `wolv`

In [ ]:
# Assign Blue Water here
# YOUR CODE HERE
blwtr = ntwk.by_sub_service(trains, SUB_SVC["blwtr"])
blwtr.shape

In [ ]:
# hidden tests are within this cell

In [ ]:
# Assign Pere Marquette here
# YOUR CODE HERE
prmrq = ntwk.by_sub_service(trains, SUB_SVC["prmrq"])
prmrq.shape

In [ ]:
# hidden tests are within this cell

In [ ]:
# Assign Wolverine here
# YOUR CODE HERE
wolv = ntwk.by_sub_service(trains, SUB_SVC["wolv"])
wolv.shape

In [ ]:
# hidden tests are within this cell


#### 3.1.2 Create chart data

In [ ]:
# List of sub-services and their mappings
sub_svcs = [
    {"sub_svc": SUB_SVC["blwtr"], "frame": blwtr, "color": COLORS["blue"], "order": 2},
    {"sub_svc": SUB_SVC["prmrq"], "frame": prmrq, "color": COLORS["amtk_red"], "order": 3},
    {"sub_svc": SUB_SVC["wolv"], "frame": wolv, "color": COLORS["amtk_blue"], "order": 1},
]

# Create a three-column DataFrame comprising average late times for each sub-service
mich_sub_svcs = pd.DataFrame({
    sub_svc["sub_svc"]: sub_svc["frame"][COLS["late_detrn_avg_mm_late"]]
    .dropna()
    .reset_index(drop=True)
    for sub_svc in sub_svcs
})

# print(mich_sub_svcs.head())

# Melt the DataFrame for charting purposes
chrt_data = pd.melt(
    mich_sub_svcs,
    var_name="Sub Service",
    value_name="Average Minutes Late",
)

# print(chrt_data.head())

# Histogram color and order mappings
hst_colors = {sub_svc["sub_svc"]: sub_svc["color"] for sub_svc in sub_svcs}
hst_order = {sub_svc["sub_svc"]: sub_svc["order"] for sub_svc in sub_svcs}

# Enforce the layering order
chrt_data["order"] = chrt_data[COLS["sub_svc"]].map(hst_order)
# chrt_data

#### 3.1.3 Generate histogram

In [ ]:
title = chrt.format_title(mich_stats, f"Amtrak {SVC['mich']} Service Late Detraining Passengers")
chart = (
    hstl_bld.LayeredHistogramBuilder(hstl.LayeredHistogramChart(frame=chrt_data, title=title))
    .x(
        axis_tick_count=mich_max_val_ceil,
        axis_values=np.arange(0.0, mich_max_val_ceil + 15, 5).tolist(),
        bin_max_bins=mich_num_bins,
        bin_step=mich_bin_width,
        scale_domain=[0.0, mich_max_val_ceil],
    )
    .color(scale_domain=list(hst_colors.keys()), scale_range=list(hst_colors.values()))
    .mu_rule(mu=mich_mu)
    .sigma_rules(sigma=mich_sigma, n=2)
    .build()
)
chart.display()

## 4.0 Michigan _Blue Water_ service (Chicago, IL - Port Huron, MI)

The _Blue Water_ operates between [Chicago Union Station](https://www.amtrak.com/stations/chi), Chicago, IL ([CHI](https://www.amtrak.com/stations/chi))
and Port Huron, MI ([PTH](https://www.amtrak.com/stations/pth)). Intermediate stops include
Kalamazoo, MI ([KAL](https://www.amtrak.com/stations/kal)),
Battle Creek, MI ([BTL](https://www.amtrak.com/stations/btl)),
East Lansing, MI ([LNS](https://www.amtrak.com/stations/lns)),
and Flint, MI ([FLN](https://www.amtrak.com/stations/fln)), among other towns and cities.

### 4.1 _Blue Water_: on-time performance metrics (entire period) [2 pts]

_Blue Water_ performance data is a compilation of quarterly metrics that focus on late
detraining passengers. Detraining assengers are considered on-time if they arrive at their
destination no later than fifteen (`15`) minutes after their scheduled arrival time. All other
detraining passengers are considered late.

Retrieve the _Blue Water_ row from the `mich_sub_svcs_stats` DataFrame. Call the appropriate
`DataFrame` method to convert the row to a `Series`. Assign the return value to a variable named
`blwtr_stats`.

In [ ]:
# YOUR CODE HERE
blwtr_stats = mich_sub_svcs_stats.set_index(COLS["sub_svc"]).loc[SUB_SVC["blwtr"]].squeeze()
blwtr_stats

In [ ]:
# hidden tests are within this cell

In [ ]:
# Total train arrivals
blwtr_trn_arrivals = blwtr_stats["Train Arrivals"]
# blwtr_trn_arrivals

# Detraining totals
blwtr_detrn = blwtr_stats[f"{COLS['total_detrn']} sum"]
blwtr_detrn_late = blwtr_stats[f"{COLS['late_detrn']} sum"]
blwtr_detrn_on_time = blwtr_detrn_late - blwtr_detrn

print(
    f"Train Arrivals: {blwtr_trn_arrivals}",
    f"Total Detraining Customers: {blwtr_detrn}",
    f"Late Detraining Customers: {blwtr_detrn_late}",
    f"On-Time Detraining Customers: {blwtr_detrn_on_time}",
    sep="\n",
)

In [ ]:
# hidden tests are within this cell

### 4.2 _Blue Water_ trains [1 pt]

Each _Blue Water_ train is identified by a unique train number.

Create a `DataFrame` named `blwtr_trns` that contains one row for each train comprising the
_Pere Marquette_ service. Include the following columns in the `DataFrame` in the order specified:

1. "Service Line"
2. "Service"
3. "Sub Service"
4. "Route Miles"
5. "Train Number"

Reset the index (set `drop=True`) when creating the new `DataFrame`.

In [ ]:
# YOUR CODE HERE
blwtr_trns = (
    blwtr[[COLS["svc_line"], COLS["svc"], COLS["sub_svc"], COLS["route_miles"], COLS["trn"]]]
    .drop_duplicates().reset_index(drop=True)
)
blwtr_trns

In [ ]:
# hidden tests are within this cell

### 4.3 _Blue Water_: mean late arrival times [2 pts]

In an earlier notebook, a _simple_ least-squares linear regression was formulated to estimate the
linear relationship between mean late arrival times and distance traveled (e.g., route miles). The
model suggests that with every additional route mile, the average minutes late for late detraining
passengers increases by approximately `0.0338` minutes. The _R_&sup2; value indicated that around
`25.5%` of the variability in the average minutes late could be explained by the number of route
miles traveled, with the remaining variability due to other factors or random noise.

For _Blue Water_ late detraining passengers traveling the entire route, the model predicts a mean
late arrival time of approximately `39.29` minutes.

Retrieve the _Blue Water_ row from the `predictions` DataFrame. Call the appropriate `DataFrame`
method to convert the row to a `Series`. Assign the return value to a variable named
`blwtr_predicted`.

In [ ]:
# YOUR CODE HERE
blwtr_predicted = predictions[predictions[COLS["sub_svc"]] == SUB_SVC["blwtr"]].squeeze()
blwtr_predicted

In [ ]:
# hidden tests are within this cell

Contrast this prediction with the actual mean late arrival times experienced by _Blue Water_ late detraining passengers.

In [ ]:
# Drop missing values
blwtr_avg_mm_late = blwtr[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Call the custom frm.describe_numeric_column() function
blwtr_avg_mm_late_describe = frm.describe_numeric_column(blwtr_avg_mm_late)
blwtr_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

### 4.4 _Blue Water_: eastbound and westbound routes [1 pt]

Stations served by eastbound and westbound _Blue Water_ trains.

In [ ]:
# Retrieve the sub service from the Amtrak sub services list
blwtr_sub_svc = next(
    (sub_svc for sub_svc in amtk_sub_svcs if sub_svc["sub service"] == SUB_SVC["blwtr"])
)
blwtr_stn_codes = blwtr_sub_svc["station codes"]
blwtr_stns = stations[stations[COLS["station_code"]].isin(blwtr_stn_codes)].reset_index(drop=True)
blwtr_stns.sort_values(by=COLS["lon"], inplace=True)
blwtr_stns

Station order for eastbound and westbound _Blue Water_ trains.

In [ ]:
blwtr_stn_order_eb = blwtr_sub_svc["station order"]["eastbound"]  # ← "station" → "station order"
blwtr_stn_order_wb = blwtr_sub_svc["station order"]["westbound"]
# blwtr_stn_order_eb, blwtr_stn_order_wb

In [ ]:
# hidden tests are within this cell

### 4.5 _Blue Water_: eastbound detraining passengers summary statistics

#### 4.5.1 _Blue Water_ Train 364 [1 pt]

In [ ]:
rte_cols = [
    COLS["trn"],          # ← WAJIB ada untuk add_stations_to_route
    COLS["station_code"],
    COLS["station"],
    COLS["state"],
    COLS["lat"],
    COLS["lon"],
]
amtk_364 = ntwk.by_train_number(trains, 364)
amtk_364_rte = ntwk.create_route(amtk_364, TRN["364"]["direction"], blwtr_stn_order_eb)
amtk_364_rte_stats = detrn.get_route_sum_stats(
    amtk_364_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_364_rte_stats

In [ ]:
# hidden tests are within this cell

##### 4.5.1.1 Write to file [1 pt]

Write `amtk_364_rte_stats` to a CSV file named `stu-amtk_364_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_364_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_364_rte_stats.csv")
amtk_364_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

### 4.6 _Blue Water_ eastbound mean late arrival times

#### 4.6.1 _Blue Water_ Train 364 [1 pt]

Review the central tendency, dispersion, and shape for the mean late arrival times of train 364.

In [ ]:
# Drop missing values
amtk_364_avg_mm_late = amtk_364[COLS["late_detrn_avg_mm_late"]].reset_index(drop=True)

# Describe the column
amtk_364_avg_mm_late_describe = frm.describe_numeric_column(amtk_364_avg_mm_late)
amtk_364_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 4.6.2 Generate box plots

##### 4.6.2.1 Assemble the chart data

In [ ]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_364_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
chrt_data

##### 4.6.2.2 Preaggregate the data

In [ ]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 4.6.2.3 Generate chart

In [ ]:
# Create chart title
title_txt = (
    f"Amtrak {TRN['364']['name']} Train {TRN['364']['number']} Late Detraining Passengers\n"
    f"{TRN['364']['route']} ({TRN['364']['direction']})"
)
title = chrt.format_title(amtk_364_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

### 4.7 _Blue Water_: visualize eastbound mean late arrival times by station

In [ ]:
title = chrt.format_title(
    amtk_364_rte_stats,
    f"Amtrak {TRN['364']['name']} Train {TRN['364']['number']} Late Detraining Passengers (2202 Q1 - 2024 Q3)",
)

chart = (
    ln_bld.LineChartBuilder(ln.LineChart(frame=amtk_364_rte_stats, title=title))
    .x(axis_label_angle=33, sort=amtk_364_rte_stats.index.tolist())
    .y(axis_tick_count_max=100, axis_values=np.arange(0.0, 100.0, 5).tolist())
    .color(scale_domain=[364], scale_range=[COLORS["amtk_blue"]], disable_legend=True)
    .build()
)
chart.display()

### 4.8 _Blue Water_: westbound detraining passengers summary statistics

#### 4.8.1 _Blue Water_ Train 365 [1 pt]

Review previous code employed to generate summary statistics for an Amtrak train. Then leverage
functions available in the `amtk_network` and `amtk_detrain` modules to create three new
`DataFrame` objects named `amtk_365`, `amtk_365_rte`, and `amtk_365_rte_stats`, respectively.

In [ ]:
# YOUR CODE HERE
amtk_365 = ntwk.by_train_number(trains, 365)
amtk_365_rte = ntwk.create_route(amtk_365, TRN["365"]["direction"], blwtr_stn_order_wb)
amtk_365_rte_stats = detrn.get_route_sum_stats(
    amtk_365_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_365_rte_stats

In [ ]:
# hidden tests are within this cell

##### 4.8.1.1 Write to file [1 pt]

Write `amtk_365_rte_stats` to a CSV file named `stu-amtk_365_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_365_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_365_rte_stats.csv")
amtk_365_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

### 4.9 _Blue Water_: westbound mean late arrival times

#### 4.9.1 _Blue Water_ Train 365 [1 pt]

Review the central tendency, dispersion, and shape for the mean late arrival times of
train 365.

In [ ]:
# Drop missing values
amtk_365_avg_mm_late = amtk_365[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_365_avg_mm_late_describe = frm.describe_numeric_column(amtk_365_avg_mm_late)
amtk_365_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 4.9.2 Generate box plots

##### 4.9.2.1 Assemble the chart data

In [ ]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_365_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
chrt_data

##### 4.9.2.2 Preaggregate the data

In [ ]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 4.9.2.3 Generate chart

In [ ]:
# Create chart title
title_txt = (
    f"Amtrak {TRN['365']['name']} Train {TRN['365']['number']} Late Detraining Passengers\n"
    f"{TRN['365']['route']} ({TRN['365']['direction']})"
)
title = chrt.format_title(amtk_365_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

### 4.10 _Blue Water_: visualize westbound mean late arrival times by station

In [ ]:
title = chrt.format_title(
    amtk_365_rte_stats,
    f"Amtrak {TRN['365']['name']} Train {TRN['365']['number']} Late Detraining Passengers (2202 Q1 - 2024 Q3)",
)

chart = (
    ln_bld.LineChartBuilder(ln.LineChart(frame=amtk_365_rte_stats, title=title))
    .x(axis_label_angle=33, sort=amtk_365_rte_stats.index.tolist())
    .y(axis_tick_count_max=100, axis_values=np.arange(0.0, 100.0, 5).tolist())
    .color(scale_domain=[365], scale_range=[COLORS["amtk_blue"]], disable_legend=True)
    .build()
)
chart.display()

## 5.0 Michigan _Pere Marquette_ service (Chicago, IL - Grand Rapids, MI)

The _Pere Marquette_ operates daily between
[Chicago Union Station](https://www.amtrak.com/stations/chi), Chicago, IL
([CHI](https://www.amtrak.com/stations/chi)) and 
Grand Rapids ([GRR](https://www.amtrak.com/stations/grr)), MI. Intermediate stops include
St. Joseph-Benton Harbor, MI ([SJM](https://www.amtrak.com/stations/sjm.html)),
Bangor, MI ([BAM](https://www.amtrak.com/stations/bam.html)), and
Holland, MI ([HOM](https://www.amtrak.com/stations/hom.html)).

### 5.1 _Pere Marquette_: on-time performance metrics (entire period) [2 pts]

_Pere Marquette_ performance data is a compilation of quarterly metrics that focus on late
detraining passengers. Detraining assengers are considered on-time if they arrive at their
destination no later than fifteen (`15`) minutes after their scheduled arrival time. All other
detraining passengers are considered late.

Retrieve the _Pere Marquette_ row from the `mich_sub_svcs_stats` DataFrame. Call the appropriate
`DataFrame` method to convert the row to a `Series`. Assign the return value to a variable named
`prmrq_stats`.

In [ ]:
# YOUR CODE HERE
prmrq_stats = mich_sub_svcs_stats.set_index(COLS["sub_svc"]).loc[SUB_SVC["prmrq"]].squeeze()
prmrq_stats

In [ ]:
# hidden tests are within this cell

In [ ]:
# Total train arrivals
prmrq_trn_arrivals = prmrq_stats["Train Arrivals"]
# prmrq_trn_arrivals

# Detraining totals
prmrq_detrn = prmrq_stats[f"{COLS['total_detrn']} sum"]
prmrq_detrn_late = prmrq_stats[f"{COLS['late_detrn']} sum"]
prmrq_detrn_on_time = prmrq_detrn - prmrq_detrn_late

print(
    f"Train Arrivals: {prmrq_trn_arrivals}",
    f"Total Detraining Customers: {prmrq_detrn}",
    f"Late Detraining Customers: {prmrq_detrn_late}",
    f"On-Time Detraining Customers: {prmrq_detrn_on_time}",
    sep="\n",
)

In [ ]:
# hidden tests are within this cell

### 5.2 _Pere Marquette_ trains [1 pt]

Each _Pere Marquette_ train is identified by a unique train number.

Create a `DataFrame` named `prmrq_trns` that contains one row for each train comprising the
_Pere Marquette_ service. Include the following columns in the `DataFrame` in the order specified:

1. "Service Line"
2. "Service"
3. "Sub Service"
4. "Route Miles"
5. "Train Number"

Reset the index (set `drop=True`) when creating the new `DataFrame`.

In [ ]:
# YOUR CODE HERE
prmrq_trns = (
    prmrq[[COLS["svc_line"], COLS["svc"], COLS["sub_svc"], COLS["route_miles"], COLS["trn"]]]
    .drop_duplicates().reset_index(drop=True)
)
prmrq_trns

In [ ]:
# hidden tests are within this cell

### 5.3 _Pere Marquette_: mean late arrival times [2 pts]

In an earlier notebook, a _simple_ least-squares linear regression was formulated to estimate the
linear relationship between mean late arrival times and distance traveled (e.g., route miles). The
model suggests that with every additional route mile, the average minutes late for late detraining
passengers increases by approximately `0.0338` minutes. The _R_&sup2; value indicated that around
`25.5%` of the variability in the average minutes late could be explained by the number of route
miles traveled, with the remaining variability due to other factors or random noise.

For _Pere Marquette_ late detraining passengers traveling the entire route, the model predicts a mean
late arrival time of approximately `34.38`` minutes.

Retrieve the _Pere Marquette_ row from the `predictions` DataFrame. Call the appropriate `DataFrame`
method to convert the row to a `Series`. Assign the return value to a variable named
`prmrq_predicted`.

In [ ]:
# YOUR CODE HERE
prmrq_predicted = predictions[predictions[COLS["sub_svc"]] == SUB_SVC["prmrq"]].squeeze()
prmrq_predicted

In [ ]:
# hidden tests are within this cell

Contrast this prediction with the actual mean late arrival times experienced by _Pere Marquette_ late detraining passengers.

In [ ]:
# Drop missing values
prmrq_avg_mm_late = prmrq[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Call the custom frm.describe_numeric_column() function
prmrq_avg_mm_late_describe = frm.describe_numeric_column(prmrq_avg_mm_late)
prmrq_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

### 5.4 _Pere Marquette_: eastbound and westbound routes [1 pt]

Stations served by eastbound and westbound _Pere Marquette_ trains.

In [ ]:
# Retrieve the sub service from the Amtrak sub services list
prmrq_sub_svc = next(
    (sub_svc for sub_svc in amtk_sub_svcs if sub_svc["sub service"] == SUB_SVC["prmrq"])  # ← blwtr→prmrq
)
prmrq_stn_codes = prmrq_sub_svc["station codes"]
prmrq_stns = stations[stations[COLS["station_code"]].isin(prmrq_stn_codes)].reset_index(drop=True)
prmrq_stns.sort_values(by=COLS["lon"], inplace=True)
prmrq_stns

In [ ]:
prmrq_stn_order_eb = prmrq_sub_svc["station order"]["eastbound"]
prmrq_stn_order_wb = prmrq_sub_svc["station order"]["westbound"]
# prmrq_stn_order_eb, prmrq_stn_order_wb

In [ ]:
# hidden tests are within this cell

### 5.5 _Pere Marquette_: eastbound detraining passengers summary statistics

#### 5.5.1 _Pere Marquette_ Train 370 [1 pt]

In [ ]:
# Base columns for routes
rte_cols = [
    COLS["trn"],
    COLS["station_code"], # Urutan ditukar
    COLS["station"],      # Urutan ditukar
    COLS["state"],
    COLS["lat"],
    COLS["lon"],
]

# Train 370 eastbound
amtk_370 = ntwk.by_train_number(trains, 370)

amtk_370_rte = ntwk.create_route(amtk_370, TRN["370"]["direction"], prmrq_stn_order_eb)

amtk_370_rte_stats = detrn.get_route_sum_stats(
    amtk_370_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_370_rte_stats

In [ ]:
# hidden tests are within this cell

##### 5.5.1.1 Write to file [1 pt]

Write `amtk_370_rte_stats` to a CSV file named `stu-amtk_370_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_370_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_370_rte_stats.csv")
amtk_370_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

### 5.6 _Pere Marquette_: eastbound mean late arrival times

#### 5.6.1 _Pere Marquette_ Train 370 [1 pt]

Review the central tendency, dispersion, and shape for the mean late arrival times of train 370.

In [ ]:
# Drop missing values
amtk_370_avg_mm_late = amtk_370[COLS["late_detrn_avg_mm_late"]].reset_index(drop=True)

# Describe the column
amtk_370_avg_mm_late_describe = frm.describe_numeric_column(amtk_370_avg_mm_late)
amtk_370_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 5.6.2 Generate box plots

##### 5.6.2.1 Assemble the chart data

In [ ]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_370_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
# chrt_data

##### 5.6.2.2 Preaggregate the data

In [ ]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 5.6.2.3 Generate chart

In [ ]:
# Create chart title
title_txt = (
    f"Amtrak {TRN['370']['name']} Train {TRN['370']['number']} Late Detraining Passengers\n"
    f"{TRN['370']['route']} ({TRN['370']['direction']})"
)
title = chrt.format_title(amtk_370_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

### 5.7 _Pere Marquette_: visualize eastbound mean late arrival times by station

In [ ]:
title = chrt.format_title(
    amtk_370_rte_stats,
    f"Amtrak {TRN['370']['name']} Train {TRN['370']['number']} Late Detraining Passengers (2202 Q1 - 2024 Q3)",
)

chart = (
    ln_bld.LineChartBuilder(ln.LineChart(frame=amtk_370_rte_stats, title=title))
    .x(sort=amtk_370_rte_stats.index.tolist())
    .y(axis_tick_count_max=60, axis_values=np.arange(0.0, 60.0, 5).tolist())
    .color(scale_domain=[370], scale_range=[COLORS["amtk_blue"]], disable_legend=True)
    .build()
)
chart.display()

### 5.8 _Pere Marquette_: westbound detraining passengers summary statistics

#### 5.8.1 _Pere Marquette_ Train 371 [1 pt]

Review previous code employed to generate summary statistics for an Amtrak train. Then leverage
functions available in the `amtk_network` and `amtk_detrain` modules to create three new
`DataFrame` objects named `amtk_371`, `amtk_371_rte`, and `amtk_371_rte_stats`, respectively.

In [ ]:
# YOUR CODE HERE
amtk_371 = ntwk.by_train_number(trains, 371)
amtk_371_rte = ntwk.create_route(amtk_371, TRN["371"]["direction"], prmrq_stn_order_wb)
amtk_371_rte_stats = detrn.get_route_sum_stats(
    amtk_371_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_371_rte_stats

In [ ]:
# hidden tests are within this cell

##### 5.8.1.1 Write to file [1 pt]

Write `amtk_371_rte_stats` to a CSV file named `stu-amtk_371_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_371_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_371_rte_stats.csv")
amtk_371_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

### 5.9 _Pere Marquette_: westbound mean late arrival times

#### 5.9.1 _Pere Marquette_ Train 371 [1 pt]

Review the central tendency, dispersion, and shape for the mean late arrival times of train 371.

In [ ]:
# Drop missing values
amtk_371_avg_mm_late = amtk_371[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_371_avg_mm_late_describe = frm.describe_numeric_column(amtk_371_avg_mm_late)
amtk_371_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 5.9.2 Generate box plots

##### 5.9.2.1 Assemble the chart data

In [ ]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_371_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
# chrt_data

##### 5.9.2.2 Preaggregate the data

In [ ]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 5.9.2.3 Generate chart

In [ ]:
# Create chart title
title_txt = (
    f"Amtrak {TRN['371']['name']} Train {TRN['371']['number']} Late Detraining Passengers\n"
    f"{TRN['371']['route']} ({TRN['371']['direction']})"
)
title = chrt.format_title(amtk_371_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

### 5.10 _Pere Marquette_: visualize westbound mean late arrival times by station

In [ ]:
title = chrt.format_title(
    amtk_371_rte_stats,
    f"Amtrak {TRN['371']['name']} Train {TRN['371']['number']} Late Detraining Passengers (2202 Q1 - 2024 Q3)",
)

chart = (
    ln_bld.LineChartBuilder(ln.LineChart(frame=amtk_371_rte_stats, title=title))
    .x(sort=amtk_371_rte_stats.index.tolist())
    .y(axis_tick_count_max=75, axis_values=np.arange(0.0, 75.0, 5).tolist())
    .color(scale_domain=[371], scale_range=[COLORS["amtk_blue"]], disable_legend=True)
    .build()
)
chart.display()

## 6.0 Michigan _Wolverine_ service (Chicago, IL - Pontiac, MI)

_Wolverine_ trains operates between
[Chicago Union Station](https://www.amtrak.com/stations/chi),
Chicago, IL ([CHI](https://www.amtrak.com/stations/chi)) and
Pontiac, MI ([PNT](https://www.amtrak.com/stations/pnt)). Intermediate stops include
Kalamazoo, MI ([KAL](https://www.amtrak.com/stations/kal)),
Battle Creek, MI ([BTL](https://www.amtrak.com/stations/btl)),
Jackson, MI ([JXN](https://www.amtrak.com/stations/jxn)),
Ann Arbor, MI ([ARB](https://www.amtrak.com/stations/arb)), and
Detroit, MI ([DET](https://www.amtrak.com/stations/det)), among other towns and cities.

### 6.1 _Wolverine_: on-time performance metrics (entire period) [2 pts]

_Wolverine_ performance data is a compilation of quarterly metrics that focus on late
detraining passengers. Detraining assengers are considered on-time if they arrive at their
destination no later than fifteen (`15`) minutes after their scheduled arrival time. All other
detraining passengers are considered late.

Retrieve the _Wolverine_ row from the `mich_sub_svcs_stats` DataFrame. Call the appropriate
`DataFrame` method to convert the row to a `Series`. Assign the return value to a variable named
`wolv_stats`.

In [ ]:
# YOUR CODE HERE
wolv_stats = mich_sub_svcs_stats.set_index(COLS["sub_svc"]).loc[SUB_SVC["wolv"]].squeeze()
wolv_stats

In [ ]:
# hidden tests are within this cell

In [ ]:
# Total train arrivals
wolv_trn_arrivals = wolv_stats["Train Arrivals"]
# wolv_trn_arrivals

# Detraining totals
wolv_detrn = wolv_stats[f"{COLS['total_detrn']} sum"]
wolv_detrn_late = wolv_stats[f"{COLS['late_detrn']} sum"]   # ← tambahkan " sum"
wolv_detrn_on_time = wolv_detrn - wolv_detrn_late

print(
    f"Train Arrivals: {wolv_trn_arrivals}",
    f"Total Detraining Customers: {wolv_detrn}",
    f"Late Detraining Customers: {wolv_detrn_late}",
    f"On-Time Detraining Customers: {wolv_detrn_on_time}",
    sep="\n",
)

In [ ]:
# hidden tests are within this cell

### 6.2 _Wolverine_ trains [1 pt]

Each _Wolverine_ train is identified by a unique train number.

Create a `DataFrame` named `wolv_trns` that contains one row for each train comprising the
_Pere Marquette_ service. Include the following columns in the `DataFrame` in the order specified:

1. "Service Line"
2. "Service"
3. "Sub Service"
4. "Route Miles"
5. "Train Number"

Reset the index (set `drop=True`) when creating the new `DataFrame`.

In [ ]:
# YOUR CODE HERE
wolv_trns = (
    wolv[[COLS["svc_line"], COLS["svc"], COLS["sub_svc"], COLS["route_miles"], COLS["trn"]]]
    .drop_duplicates().reset_index(drop=True)
)
wolv_trns

In [ ]:
# hidden tests are within this cell

### 6.3 _Wolverine_: mean late arrival times [2 pts]

In an earlier notebook, a _simple_ least-squares linear regression was formulated to estimate the
linear relationship between mean late arrival times and distance traveled (e.g., route miles). The
model suggests that with every additional route mile, the average minutes late for late detraining
passengers increases by approximately `0.0338` minutes. The _R_&sup2; value indicated that around
`25.5%` of the variability in the average minutes late could be explained by the number of route
miles traveled, with the remaining variability due to other factors or random noise.

For _Wolverine_ late detraining passengers traveling the entire route, the model predicts a mean
late arrival time of approximately `38.61` minutes.

Retrieve the _Wolverine_ row from the `predictions` DataFrame. Call the appropriate `DataFrame`
method to convert the row to a `Series`. Assign the return value to a variable named
`wolv_predicted`.

In [ ]:
# YOUR CODE HERE
wolv_predicted = predictions[predictions[COLS["sub_svc"]] == SUB_SVC["wolv"]].squeeze()
wolv_predicted

In [ ]:
# hidden tests are within this cell

Contrast this prediction with the actual mean late arrival times experienced by _Wolverine_ late detraining passengers.

In [ ]:
# Drop missing values
wolv_avg_mm_late = wolv[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Call the custom frm.describe_numeric_column() function again
wolv_avg_mm_late_describe = frm.describe_numeric_column(wolv_avg_mm_late)
wolv_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

### 6.4 _Wolverine_: eastbound and westbound routes [1 pt]

Stations served by eastbound and westbound _Wolverine_ trains.

In [ ]:
# Retrieve the sub service from the Amtrak sub services list
wolv_sub_svc = next(
    (sub_svc for sub_svc in amtk_sub_svcs if sub_svc["sub service"] == SUB_SVC["wolv"])
)
wolv_stn_codes = wolv_sub_svc["station codes"]
wolv_stns = stations[stations[COLS["station_code"]].isin(wolv_stn_codes)].reset_index(drop=True)
# WARN: longitude sort does not guarantee correct station order: ROY, TRM, PNT last/first 3 stops
wolv_stns.sort_values(by=COLS["lon"], inplace=True)
wolv_stns

Station order for eastbound and westbound _Pere Marquette_ trains.

In [ ]:
# Correct order of stations for eastbound and westbound trains
wolv_stn_order_eb = wolv_sub_svc["station order"]["eastbound"]  # ← "eb" → "eastbound"
wolv_stn_order_wb = wolv_sub_svc["station order"]["westbound"]  # ← "wb" → "westbound"
# wolv_stn_order_eb, wolv_stn_order_wb

In [ ]:
# hidden tests are within this cell

### 6.5 _Wolverine_: eastbound detraining passengers summary statistics [2 pts]

Call the appropriate `DataFrame` method to retrieve the station data for trains 350, 352, and 354
referenced in `wolv`. Assign the new `DataFrame` returned by the method call to a variable named
`wolv_eb`.

In [ ]:
# YOUR CODE HERE
wolv_eb = wolv[wolv[COLS["trn"]].isin([350, 352, 354])].reset_index(drop=True)
wolv_eb

In [ ]:
# hidden tests are within this cell

Compute the summary statistics for westbound _Wolverine_ trains. The `amtk_detrain` module includes
a function that you can call to perform the computation. Pass to it `wolv_eb` along with other
arguments required by the function. Assign the return value of the function call to a variable named
`wolv_eb_stats`.

In [ ]:
# YOUR CODE HERE
wolv_eb_stats = detrn.get_sum_stats(wolv_eb, AGG["columns"], AGG["funcs"])
wolv_eb_stats["Late Detraining Customers Avg Min Late mean"] = round(wolv_eb[COLS["late_detrn_avg_mm_late"]].mean(), 4)
wolv_eb_stats

In [ ]:
# hidden tests are within this cell

#### 6.5.1 _Wolverine_ Train 350 [1 pt]

In [ ]:
rte_cols = [
    COLS["trn"],
    COLS["station_code"],
    COLS["station"],
    COLS["state"],
    COLS["lat"],
    COLS["lon"],
]

# Train 350 eastbound
amtk_350 = ntwk.by_train_number(trains, 350)
amtk_350_rte = ntwk.create_route(amtk_350, TRN["350"]["direction"], wolv_stn_order_eb)
amtk_350_rte_stats = detrn.get_route_sum_stats(
    amtk_350_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_350_rte_stats

In [ ]:
# hidden tests are within this cell

##### 6.5.1.1 Write to file [1 pt]

Write `amtk_350_rte_stats` to a CSV file named `stu-amtk_350_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_350_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_350_rte_stats.csv")
amtk_350_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

#### 6.5.2 _Wolverine_ Train 352 [1 pt]

In [ ]:
amtk_352 = ntwk.by_train_number(trains, 352)
amtk_352_rte = ntwk.create_route(amtk_352, TRN["352"]["direction"], wolv_stn_order_eb)
amtk_352_rte_stats = detrn.get_route_sum_stats(
    amtk_352_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_352_rte_stats

In [ ]:
# hidden tests are within this cell

##### 6.5.2.1 Write to file [1 pt]

Write `amtk_352_rte_stats` to a CSV file named `stu-amtk_352_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_352_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_352_rte_stats.csv")
amtk_352_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

#### 6.5.3 _Wolverine_ Train 354 [1 pt]

In [ ]:
# Train 354 eastbound
amtk_354 = ntwk.by_train_number(trains, 354)
amtk_354_rte = ntwk.create_route(amtk_354, TRN["354"]["direction"], wolv_stn_order_eb)
amtk_354_rte_stats = detrn.get_route_sum_stats(
    amtk_354_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_354_rte_stats

In [ ]:
# hidden tests are within this cell

##### 6.5.3.1 Write to file [1 pt]

Write `amtk_354_rte_stats` to a CSV file named `stu-amtk_354_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_354_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_354_rte_stats.csv")
amtk_354_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

### 6.6 _Wolverine_: eastbound mean late arrival times

Review the central tendency, dispersion, and shape for the mean late arrival times of trains 350, 352, and 354.

#### 6.6.1 _Wolverine_ Train 350 [1 pt]

In [ ]:
# Drop missing values
amtk_350_avg_mm_late = amtk_350[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)  # ← tambahkan dropna

# Describe the column
amtk_350_avg_mm_late_describe = frm.describe_numeric_column(amtk_350_avg_mm_late)
amtk_350_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 6.6.2 _Wolverine_ Train 352 [1 pt]

In [ ]:
# Drop missing values
amtk_352_avg_mm_late = amtk_352[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_352_avg_mm_late_describe = frm.describe_numeric_column(amtk_352_avg_mm_late)
amtk_352_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 6.6.3 _Wolverine_ Train 354 [1 pt]

In [ ]:
# Drop missing values
amtk_354_avg_mm_late = amtk_354[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_354_avg_mm_late_describe = frm.describe_numeric_column(amtk_354_avg_mm_late)
amtk_354_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 6.6.4 Generate box plots

Retrieve the chart data, preaggregate it, and generate the box plots.

In [ ]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Data for boxplots
wolv_eb_trns = [
    {"number": 350, "route": amtk_350_rte, "stats": amtk_350_rte_stats},
    {"number": 352, "route": amtk_352_rte, "stats": amtk_352_rte_stats},
    {"number": 354, "route": amtk_354_rte, "stats": amtk_354_rte_stats},
]

# Assemble charts
for trn in wolv_eb_trns:
    chrt_data = detrn.get_qtr_avg_min_late(
        trn["route"], cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
    )

    # Pre-aggregate the data
    chrt_data = frm.aggregate_data(
        chrt_data, [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]
    )

    # Create chart title
    trn_txt = TRN[str(trn["number"])]
    title_txt = (
        f"Amtrak {trn_txt['name']} Train {trn_txt['number']} Late Detraining Passengers\n"
        f"{trn_txt['route']} ({trn_txt['direction']})"
    )
    title = chrt.format_title(trn["stats"], title_txt)

    # Generate chart
    chart = (
        bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
        .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
        .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
        .build()
    )
    chart.display()

### 6.7 _Wolverine_: visualize eastbound mean late arrival times by station

Visualizing mean late arrival times by station for each train in a single chart requires aligning
the station order for trains 350, 352, and 354. This requires adding "placeholder" rows to each
train `DataFrame` that represent stations not visited by all trains.

| Train | "Placeholder" Station(s) | Notes |
| :---- | :----------------------- | :---- |
| 350   | Albion, MI ([ALI](https://www.amtrak.com/stations/ali)) | &nbsp; |
| 352   | Albion, MI ([ALI](https://www.amtrak.com/stations/ali)), Dowagiac, MI ([DOA](https://www.amtrak.com/stations/doa)), Hammond-Whiting, IN ([HMI](https://www.amtrak.com/stations/hmi)), Michigan City, IN ([MCI](https://en.wikipedia.org/wiki/Michigan_City_station)) | MCI closed 4 April 2022 |
| 354   | Dowagiac, MI ([DOA](https://www.amtrak.com/stations/doa)), Hammond-Whiting, IN ([HMI](https://www.amtrak.com/stations/hmi)) | &nbsp; |

#### 6.7.1 Reindex `wolv_stns`

In [ ]:
# Reindex DataFrame based on amtk_350_rte_stats columns.
wolv_stns = wolv_stns.reindex(columns=amtk_350_rte_stats.columns).reset_index(drop=True)
# wolv_stns

#### 6.7.2 Assemble chart data [1 pt]

Combine the three `amtk_*_chrt_data` `DataFrame` objects created below by calling the function
`ntwk.add_stations_to_route()` into a single `DataFrame` object named `chrt_data`. When combining
the `DataFrame` objects ignore their current indices.

In [ ]:
amtk_350_chrt_data = ntwk.add_stations_to_route(
    amtk_350_rte_stats.copy(),
    wolv_stns[wolv_stns[COLS["station_code"]] == "ALI"],
    wolv_stn_order_eb,
)
# amtk_350_chrt_data

In [ ]:
amtk_352_chrt_data = ntwk.add_stations_to_route(
    amtk_352_rte_stats.copy(),
    wolv_stns[wolv_stns[COLS["station_code"]].isin(("ALI", "DOA", "HMI", "MCI"))],
    wolv_stn_order_eb,
)
# amtk_352_chrt_data

In [ ]:
amtk_354_chrt_data = ntwk.add_stations_to_route(
    amtk_354_rte_stats.copy(),
    wolv_stns[wolv_stns[COLS["station_code"]].isin(("DOA", "HMI"))],
    wolv_stn_order_eb,
)
# amtk_354_chrt_data

In [ ]:
# YOUR CODE HERE
chrt_data = pd.concat([amtk_350_chrt_data, amtk_352_chrt_data, amtk_354_chrt_data], ignore_index=True)


In [ ]:
# hidden tests are within this cell

#### 6.7.3 Generate line chart

In [ ]:
# Chart title
title = chrt.format_title(
    wolv_eb_stats, f"Amtrak {SUB_SVC['wolv']} Service Late Detraining Passengers"
)

# Custom line colors
line_colors = {350: COLORS["amtk_blue"], 352: COLORS["amtk_red"], 354: COLORS["blue"]}

chart = (
    ln_bld.LineChartBuilder(ln.LineChart(frame=chrt_data, title=title), interpolate=True)
    .line_mark_properties(stroke_dash=[5, 5])
    .x(axis_label_angle=33, sort=chrt_data.index.tolist())
    .y(axis_tick_count_max=100, axis_values=np.arange(0.0, 100.0, 5).tolist())
    .color(
        scale_domain=list(line_colors.keys()),
        scale_range=list(line_colors.values()),
        title="Train Number",
    )
    .build()
)
chart.display()

### 6.8 _Wolverine_: westbound detraining passengers summary statistics [2 pts]

Call the appropriate `DataFrame` method to retrieve the station data for trains 351, 353, and 355
referenced in `wolv`. Assign the new `DataFrame` returned by the method call to a variable named
`wolv_wb`.

In [ ]:
# YOUR CODE HERE
wolv_wb = wolv[wolv[COLS["trn"]].isin([351, 353, 355])].reset_index(drop=True)
wolv_wb

In [ ]:
# hidden tests are within this cell

Compute the summary statistics for westbound _Wolverine_ trains. The `amtk_detrain` module includes
a function that you can call to perform the computation. Pass to it `wolv_wb` along with other
arguments required by the function. Assign the return value of the function call to a variable named
`wolv_wb_stats`.

In [ ]:
# YOUR CODE HERE
wolv_wb_stats = detrn.get_sum_stats(wolv_wb, AGG["columns"], AGG["funcs"])
wolv_wb_stats["Late Detraining Customers Avg Min Late mean"] = round(wolv_wb[COLS["late_detrn_avg_mm_late"]].mean(), 4)
wolv_wb_stats

In [ ]:
# hidden tests are within this cell

#### 6.8.1 _Wolverine_ Train 351 [1 pt]

In [ ]:
amtk_351 = ntwk.by_train_number(trains, 351)
amtk_351_rte = ntwk.create_route(amtk_351, TRN["351"]["direction"], wolv_stn_order_wb)  # ← eb→wb
amtk_351_rte_stats = detrn.get_route_sum_stats(
    amtk_351_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_351_rte_stats

In [ ]:
# hidden tests are within this cell

##### 6.8.1.1 Write to file [1 pt]

Write `amtk_351_rte_stats` to a CSV file named `stu-amtk_351_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_351_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_351_rte_stats.csv")
amtk_351_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

#### 6.8.2 _Wolverine_ Train 353 [1 pt]

In [ ]:
amtk_353 = ntwk.by_train_number(trains, 353)    # ← hapus [COLS["trn"]]
amtk_353_rte = ntwk.create_route(amtk_353, TRN["353"]["direction"], wolv_stn_order_wb)
amtk_353_rte_stats = detrn.get_route_sum_stats(
    amtk_353_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_353_rte_stats

In [ ]:
# hidden tests are within this cell

##### 6.8.2.1 Write to file [1 pt]

Write `amtk_353_rte_stats` to a CSV file named `stu-amtk_353_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_353_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_353_rte_stats.csv")
amtk_353_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

#### 6.8.3 _Wolverine_ Train 355 [1 pt]

In [ ]:
amtk_355 = ntwk.by_train_number(trains, 355)
amtk_355_rte = ntwk.create_route(amtk_355, TRN["355"]["direction"], wolv_stn_order_wb)  # ← eb→wb
amtk_355_rte_stats = detrn.get_route_sum_stats(
    amtk_355_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols
)
amtk_355_rte_stats

In [ ]:
# hidden tests are within this cell

##### 6.8.3.1 Write to file [1 pt]

Write `amtk_355_rte_stats` to a CSV file named `stu-amtk_355_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_355_rte_stats.csv` file.
It must match line for line, character for character.

In [ ]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_355_rte_stats.csv")
amtk_355_rte_stats.to_csv(filepath, index=False)

In [ ]:
# hidden tests are within this cell

### 6.9 _Wolverine_: westbound mean late arrival times

Review the central tendency, dispersion, and shape for the mean late arrival times of trains 351, 353, and 355.

#### 6.9.1 _Wolverine_ Train 351 [1 pt]

In [ ]:
# Drop missing values
amtk_351_avg_mm_late = amtk_351[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_351_avg_mm_late_describe = frm.describe_numeric_column(amtk_351_avg_mm_late)
amtk_351_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 6.9.2 _Wolverine_ Train 353 [1 pt]

In [ ]:
# Drop missing values
amtk_353_avg_mm_late = amtk_353[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_353_avg_mm_late_describe = frm.describe_numeric_column(amtk_353_avg_mm_late)
amtk_353_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 6.9.3 _Wolverine_ Train 355 [1 pt]

In [ ]:
# Drop missing values
amtk_355_avg_mm_late = amtk_355[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_355_avg_mm_late_describe = frm.describe_numeric_column(amtk_355_avg_mm_late)
amtk_355_avg_mm_late_describe

In [ ]:
# hidden tests are within this cell

#### 6.9.4 Generate box plots

Retrieve the chart data, preaggregate it, and generate the box plots.

In [ ]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Data for boxplots
wolv_wb_trns = [
    {"number": 351, "route": amtk_351_rte, "stats": amtk_351_rte_stats},
    {"number": 353, "route": amtk_353_rte, "stats": amtk_353_rte_stats},
    {"number": 355, "route": amtk_355_rte, "stats": amtk_355_rte_stats},
]

# Assemble charts
for trn in wolv_wb_trns:
    chrt_data = detrn.get_qtr_avg_min_late(
        trn["route"], cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
    )

    # Pre-aggregate the data
    chrt_data = frm.aggregate_data(
        chrt_data, [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]
    )

    # Create chart title
    trn_txt = TRN[str(trn["number"])]
    title_txt = (
        f"Amtrak {trn_txt['name']} Train {trn_txt['number']} Late Detraining Passengers\n"
        f"{trn_txt['route']} ({trn_txt['direction']})"
    )
    title = chrt.format_title(trn["stats"], title_txt)

    # Generate chart
    chart = (
        bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
        .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
        .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
        .build()
    )
    chart.display()

### 6.10 _Wolverine_: visualize westbound mean late arrival times by station

Visualizing mean late arrival times by station for each train in a single chart requires aligning
the station order for trains 351, 353, and 355. This requires adding "placeholder" rows to each
train `DataFrame` that represent stations not visited by all trains.

| Train | "Placeholder" Station(s) | Notes |
| :---- | :----------------------- | :---- |
| 351   | Dowagiac, MI ([DOA](https://www.amtrak.com/stations/doa)), Hammond-Whiting, IN ([HMI](https://www.amtrak.com/stations/hmi)), Michigan City, IN ([MCI](https://en.wikipedia.org/wiki/Michigan_City_station)), New Buffalo, MI ([NBU](https://www.amtrak.com/stations/nbu)), Niles, MI ([NLS](https://www.amtrak.com/stations/nls)) | MCI closed 4 April 2022 |
| 353   | Albion, MI ([ALI](https://www.amtrak.com/stations/ali)), Dowagiac, MI ([DOA](https://www.amtrak.com/stations/doa)), Michigan City, IN ([MCI](https://en.wikipedia.org/wiki/Michigan_City_station)) | MCI closed 4 April 2022 |
| 355   | Albion, MI ([ALI](https://www.amtrak.com/stations/ali)) | &nbsp; |

#### 6.10.1 Assemble chart data [1 pt]

Combine the three `amtk_*_chrt_data` `DataFrame` objects created below by calling the function
`ntwk.add_stations_to_route()` into a single `DataFrame` object named `chrt_data`. When combining
the `DataFrame` objects ignore their current indices.

In [ ]:
amtk_351_chrt_data = ntwk.add_stations_to_route(
    amtk_351_rte_stats.copy(),
    wolv_stns[wolv_stns[COLS["station_code"]].isin(("DOA", "HMI", "MCI", "NBU", "NLS"))],
    wolv_stn_order_wb,
)
# amtk_351_chrt_data

In [ ]:
amtk_353_chrt_data = ntwk.add_stations_to_route(
    amtk_353_rte_stats.copy(),
    wolv_stns[wolv_stns[COLS["station_code"]].isin(("ALI", "DOA", "MCI"))],
    wolv_stn_order_wb,
)
# amtk_353_chrt_data

In [ ]:
# Add ALI to Train 355 route
amtk_355_chrt_data = ntwk.add_stations_to_route(
    amtk_355_rte_stats.copy(),
    wolv_stns[wolv_stns[COLS["station_code"]] == "ALI"],
    wolv_stn_order_wb,
)
# amtk_355_chrt_data

In [ ]:
# YOUR CODE HERE
chrt_data = pd.concat([amtk_351_chrt_data, amtk_353_chrt_data, amtk_355_chrt_data], ignore_index=True)

In [ ]:
# hidden tests are within this cell

#### 6.10.2 Generate line chart

In [ ]:
# Chart title
title = chrt.format_title(
    wolv_wb_stats, f"Amtrak {SUB_SVC['wolv']} Service Late Detraining Passengers"
)

# Custom line colors
line_colors = {351: COLORS["amtk_blue"], 353: COLORS["amtk_red"], 355: COLORS["blue"]}

chart = (
    ln_bld.LineChartBuilder(ln.LineChart(frame=chrt_data, title=title), interpolate=True)
    .line_mark_properties(stroke_dash=[5, 5])
    .x(axis_label_angle=33, sort=chrt_data.index.tolist())
    .y(axis_tick_count_max=85, axis_values=np.arange(0.0, 85.0, 5).tolist())
    .color(scale_domain=list(line_colors.keys()), scale_range=list(line_colors.values()))
    .build()
)
chart.display()

## 5.0 Watermark

In [ ]:
%load_ext watermark
%watermark -h -i -iv -m -v